In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl
import torch

from fart.model.nbeats_dataset import build_return_windows
from fart.model.nbeats_persistence import load_model
from fart.model.train_model import prepare_training_data
from fart.utils import get_latest_model_filepath, get_project_root
from fart.visualization.confidence_calibration import plot_confidence_calibration
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
artifacts_dir = get_project_root() / "artifacts"
market, interval = "BTC-EUR", "1d"

model_path = get_latest_model_filepath(artifacts_dir, market, interval)
model, config = load_model(model_path)
lookback = config.lookback

model_path.name

In [ ]:
X_train, X_test, y_train, y_test = prepare_training_data(
    data_dir=assets_dir,
    market=market,
    interval=interval,
    months=None,
)

n_train = y_train.shape[0]
close_prices = pl.concat([y_train, y_test])
X_all, y_all = build_return_windows(close_prices, lookback)

n_train_windows = max(0, n_train - lookback - 1)
X_test_windows = X_all[n_train_windows:]
y_test_windows = y_all[n_train_windows:]

In [ ]:
model.eval()
with torch.no_grad():
    mu, log_sigma = model(X_test_windows).unbind(-1)

confidence = (1 / (1 + log_sigma.exp())).numpy()
error = (y_test_windows - mu).abs().numpy()

In [ ]:
plot_confidence_calibration(confidence, error)